[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/10_decision_trees/05_failure_cases_and_interpretation.ipynb)

# Bài 05 — Failure case và diễn giải Decision Tree

**🎯 Mục tiêu**
- Khi nào cây thất bại: ranh giới chéo, quan hệ tuyến tính mượt, ngoại suy, feature nhiều giá trị (high cardinality).
- Diễn giải cây: luật if/else, decision path cho một mẫu, feature importance — và **bẫy** của MDI importance.
- Ưu điểm thực tế: không cần scaling, xử lý missing (sklearn ≥ 1.3), feature phân loại.
- Hoàn thiện template model cho Decision Tree.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/10_decision_trees

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from ml_course.visualization import plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Failure case: ranh giới chéo

Quy luật đơn giản $x_1 + x_2 > 0$ — một đường chéo. Cây chỉ cắt song song trục → phải xấp xỉ bằng **bậc thang** với rất nhiều split:

In [ ]:
X = rng.uniform(-2, 2, (600, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
for ax, (name, model) in zip(axes, [("Cây depth 3", DecisionTreeClassifier(max_depth=3)),
                                    ("Cây không giới hạn", DecisionTreeClassifier(random_state=0)),
                                    ("Logistic Regression", LogisticRegression())]):
    acc = cross_val_score(model, X, y, cv=5).mean()
    model.fit(X, y)
    extra = f", {model.get_n_leaves()} lá" if hasattr(model, "get_n_leaves") else ""
    plot_decision_boundary(model, X, y, ax=ax)
    ax.set_title(f"{name}: acc CV = {acc:.3f}{extra}")
plt.show()

**Khắc phục:** thêm feature tổ hợp ($x_1 + x_2$, tỉ số, hiệu), dùng PCA/rotation trước, hoặc model tuyến tính / ensemble.

In [ ]:
X_aug = np.c_[X, X[:, 0] + X[:, 1]]
print(f"Cây depth 3 + feature x1 + x2: acc CV = {cross_val_score(DecisionTreeClassifier(max_depth=3), X_aug, y, cv=5).mean():.3f}")

## 2. Failure case: quan hệ tuyến tính mượt + ít dữ liệu

Với $y = 3x_1 - 2x_2 + \varepsilon$, model tuyến tính học đúng ngay với vài chục mẫu; cây phải xấp xỉ mặt phẳng bằng các khối hộp → cần rất nhiều dữ liệu:

In [ ]:
from sklearn.linear_model import LinearRegression

for n in [50, 200, 1000, 5000]:
    Xr = rng.normal(size=(n, 2))
    yr = 3 * Xr[:, 0] - 2 * Xr[:, 1] + rng.normal(0, 0.5, n)
    r2_tree = cross_val_score(DecisionTreeRegressor(min_samples_leaf=5, random_state=0), Xr, yr, cv=5, scoring="r2").mean()
    r2_lin = cross_val_score(LinearRegression(), Xr, yr, cv=5, scoring="r2").mean()
    print(f"n = {n:5d}: R² cây = {r2_tree:.3f} | R² tuyến tính = {r2_lin:.3f}")

## 3. Bẫy diễn giải: MDI importance thiên vị feature nhiều giá trị

Thêm hai cột **hoàn toàn ngẫu nhiên** vào dữ liệu churn: `random_id` (liên tục, mọi giá trị khác nhau) và `random_flag` (nhị phân). Cây không giới hạn có **rất nhiều cơ hội** chia trên `random_id` để "giải thích" nhiễu → MDI cho nó importance cao!

In [ ]:
from ml_course.datasets import make_customer_churn

churn = make_customer_churn(n=3000, seed=3)
Xc = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
Xc["random_id"] = rng.random(len(Xc))
Xc["random_flag"] = rng.integers(0, 2, len(Xc)).astype(float)
yc = churn["churn"].to_numpy()
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)

trees = {"cây không giới hạn": DecisionTreeClassifier(random_state=0),
         "cây regularize (min_samples_leaf=20)": DecisionTreeClassifier(min_samples_leaf=20, random_state=0)}
tables = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for row, (name, tree) in zip(axes, trees.items()):
    tree.fit(Xc_tr, yc_tr)
    mdi = pd.Series(tree.feature_importances_, index=Xc.columns)
    perm = permutation_importance(tree, Xc_te, yc_te, n_repeats=20, random_state=0, scoring="roc_auc")
    perm_s = pd.Series(perm.importances_mean, index=Xc.columns)
    mdi.sort_values().tail(8).plot.barh(ax=row[0], title=f"{name} — MDI (train)")
    perm_s.sort_values().tail(8).plot.barh(ax=row[1], title=f"{name} — permutation (test, ROC-AUC)")
    for ax in row:
        for label in ax.get_yticklabels():
            if label.get_text().startswith("random"):
                label.set_color("red")
    tables[name] = pd.DataFrame({"hạng MDI": mdi.rank(ascending=False).astype(int),
                                 "permutation": perm_s.round(4), "± std": perm.importances_std.round(4),
                                 "hạng perm": perm_s.rank(ascending=False).astype(int)}).loc[["random_id", "random_flag"]]
plt.tight_layout()
plt.show()
pd.concat(tables, names=["cây", "feature"])

- **MDI** tính trên dữ liệu **train** → thưởng cho mọi split giảm impurity, kể cả split "học thuộc" nhiễu; thiên vị feature liên tục / nhiều giá trị. `random_id` được xếp hạng cao ở **cả hai** cây.
- **Permutation importance** đo trên **test**: xáo trộn một cột, model tệ đi bao nhiêu. Với cây **overfit**, `random_id` vẫn còn importance dương — vì chính model đã dựa vào nó để định tuyến mẫu (model tệ → importance phản ánh cái tệ đó). Với cây **được regularize**, importance của `random_id` về gần 0, trong khoảng ± std.
- Bài học: (1) ưu tiên permutation importance trên dữ liệu held-out; (2) **sửa model trước khi diễn giải** — importance của một model overfit mô tả model đó, không mô tả dữ liệu; (3) cả hai đều là liên hệ trong model, không phải nhân quả (chương 36).

## 4. Diễn giải một dự đoán: decision path

In [ ]:
from sklearn.tree import export_text

small = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=0).fit(Xc_tr.drop(columns=["random_id", "random_flag"]), yc_tr)
names = list(Xc_tr.drop(columns=["random_id", "random_flag"]).columns)
print(export_text(small, feature_names=names, show_weights=True))

customer = Xc_te.drop(columns=["random_id", "random_flag"]).iloc[[0]]
node_ids = small.decision_path(customer).indices
print("Đường đi của khách hàng đầu tiên trong tập test:")
for node in node_ids:
    f, t = small.tree_.feature[node], small.tree_.threshold[node]
    if f >= 0:
        value = customer.iloc[0, f]
        print(f"  node {node}: {names[f]} = {value:.2f} {'<=' if value <= t else '>'} {t:.2f}")
    else:
        print(f"  lá {node}: P(churn) = {small.predict_proba(customer)[0, 1]:.2f}")

## 5. Ưu điểm thực tế

- **Không cần scaling** — split chỉ phụ thuộc thứ tự giá trị.
- **Missing values**: scikit-learn ≥ 1.3 tự học hướng đi cho giá trị thiếu ở mỗi split (với `splitter="best"`).
- Bất biến với biến đổi đơn điệu (log, sqrt...) của từng feature.
- Tự bắt **tương tác** (split con phụ thuộc split cha) và quan hệ phi tuyến.

In [ ]:
X_missing = Xc_tr.drop(columns=["random_id", "random_flag"]).copy()
X_missing.loc[X_missing.sample(frac=0.2, random_state=0).index, "tenure_months"] = np.nan
m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_missing, yc_tr)
print("train được với 20% missing ở tenure_months — accuracy test:",
      round(m.score(Xc_te.drop(columns=["random_id", "random_flag"]), yc_te), 3))

## 6. Template model: Decision Tree

| Phần | Decision Tree (CART) |
|---|---|
| Problem | phân loại, hồi quy |
| Intuition | chuỗi câu hỏi có/không chia không gian thành các vùng thuần nhất |
| Geometry | vùng quyết định là các hộp chữ nhật song song trục |
| Probability | lá ước lượng $P(y \mid \text{vùng})$ bằng tần suất |
| Objective | giảm impurity (Gini/entropy/MSE) tham lam ở mỗi node; post-pruning tối thiểu $R(T) + \alpha|T|$ |
| Optimization | greedy, đệ quy; không đảm bảo tối ưu toàn cục (bài toán tối ưu cây là NP-hard) |
| Assumptions | rất ít — không giả định dạng hàm, không cần phân phối |
| Complexity | train ~$O(d\,n\log^2 n)$; dự đoán $O(\text{độ sâu})$ |
| Hyperparameters | `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_leaf_nodes`, `ccp_alpha`, `criterion`, `class_weight` |
| Preprocessing | không cần scaling; encode feature phân loại; xử lý missing (hoặc để sklearn ≥ 1.3 tự xử lý) |
| Strengths | dễ diễn giải (cây nhỏ), phi tuyến + tương tác tự nhiên, nhanh, bất biến biến đổi đơn điệu, dữ liệu hỗn hợp |
| Weaknesses | overfit, variance cao/không ổn định, ranh giới bậc thang, không ngoại suy, xác suất thô, MDI thiên lệch |
| Interpretation | luật if/else, decision path, feature importance (ưu tiên permutation) |

## 🧠 Kiểm tra nhanh

1. Vì sao cây cần rất nhiều split để học ranh giới $x_1 + x_2 > 0$? Cách khắc phục?
2. Vì sao MDI importance cho `random_id` cao? Vì sao permutation importance trên test không bị lỗi đó?
3. Cây có cần chuẩn hoá feature không? Có bị ảnh hưởng khi lấy log một feature không?
4. Khi nào Logistic Regression tốt hơn cây quyết định?

## 📌 Tóm tắt

- Cây yếu với ranh giới chéo, quan hệ tuyến tính mượt, ngoại suy; mạnh với tương tác, phi tuyến, dữ liệu hỗn hợp.
- Diễn giải: luật, decision path; feature importance nên dùng permutation (trên dữ liệu held-out).
- Một cây đơn lẻ hiếm khi là model tốt nhất — nhưng là **viên gạch** của Random Forest và Gradient Boosting.

## 📚 Tài liệu tham khảo

- Strobl et al. (2007) — *Bias in random forest variable importance measures*
- scikit-learn User Guide — Decision Trees; Permutation feature importance
- Molnar — *Interpretable Machine Learning*, chương Decision Tree